In [ ]:
import matplotlib.pyplot as plt
from data_loader import load_glass
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
import numpy as np
import time


In [ ]:
glass = load_glass("glass")
X, y = glass["x"], glass["y"]
feature_names = glass.get("feature_names", None)
class_names = glass.get("class_names", None)

print("X:", X.shape, "y:", y.shape)
print("classes (mapped):", np.unique(y))
if class_names is not None:
    print("class_names:", class_names)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=0,
    stratify=y,
)
print(X_train.shape, X_test.shape)


Cross-validation strategy

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
results_glass = []

Grid search for k-Nearest Neighbors (kNN)

In [ ]:
knn_param_grid = {
    "n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
    "metric": ["euclidean", "manhattan"],
    "weights": ["uniform", "distance"],
}

knn = KNeighborsClassifier()

knn_grid = GridSearchCV(
    estimator=knn,
    param_grid=knn_param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1
)

start = time.perf_counter()
knn_grid.fit(X_train, y_train)
knn_time = time.perf_counter() - start

print("Best kNN params:", knn_grid.best_params_)
print("Best CV accuracy:", knn_grid.best_score_)
print(f"Grid search train time: {knn_time:.4f} sec")


In [ ]:
best_knn = knn_grid.best_estimator_

y_pred_knn = best_knn.predict(X_test)
knn_test_acc = accuracy_score(y_test, y_pred_knn)
knn_test_f1_macro = f1_score(y_test, y_pred_knn, average="macro")
knn_test_f1_weighted = f1_score(y_test, y_pred_knn, average="weighted")

print("\n=== kNN on glass test set ===")
print(f"Test accuracy     : {knn_test_acc:.4f}")
print(f"Test F1 (macro)   : {knn_test_f1_macro:.4f}")
print(f"Test F1 (weighted): {knn_test_f1_weighted:.4f}")
print("\nkNN classification report:")
print(classification_report(y_test, y_pred_knn))

cm_knn = confusion_matrix(y_test, y_pred_knn)
print("kNN confusion matrix:\n", cm_knn)

results_glass.append({
    "name": f"GLASS_kNN_best_{knn_grid.best_params_}",
    "train_time": knn_time,
    "test_accuracy": knn_test_acc,
    "test_f1_macro": knn_test_f1_macro,
    "test_f1_weighted": knn_test_f1_weighted,
})


In [ ]:
# visualize confusion matrix
plt.figure(figsize=(6, 5))
plt.imshow(cm_knn, interpolation="nearest")
plt.title("kNN Confusion Matrix (Glass)")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.colorbar()
plt.tight_layout()
plt.show()


Grid search for Nearest Class Centroid (NCC)

In [ ]:
ncc_param_grid = {
    "metric": ["euclidean", "manhattan"],
    "shrink_threshold": [None, 0.05, 0.1, 0.5, 1.0],
}

ncc = NearestCentroid()

ncc_grid = GridSearchCV(
    estimator=ncc,
    param_grid=ncc_param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1
)

start = time.perf_counter()
ncc_grid.fit(X_train, y_train)
ncc_time = time.perf_counter() - start

print("Best NCC params:", ncc_grid.best_params_)
print("Best CV accuracy:", ncc_grid.best_score_)
print(f"Grid search train time: {ncc_time:.4f} sec")


In [ ]:
best_ncc = ncc_grid.best_estimator_

y_pred_ncc = best_ncc.predict(X_test)
ncc_test_acc = accuracy_score(y_test, y_pred_ncc)
ncc_test_f1_macro = f1_score(y_test, y_pred_ncc, average="macro")
ncc_test_f1_weighted = f1_score(y_test, y_pred_ncc, average="weighted")

print("\n=== Nearest Class Centroid on glass test set ===")
print(f"Test accuracy     : {ncc_test_acc:.4f}")
print(f"Test F1 (macro)   : {ncc_test_f1_macro:.4f}")
print(f"Test F1 (weighted): {ncc_test_f1_weighted:.4f}")
print("\nNCC classification report:")
print(classification_report(y_test, y_pred_ncc))

cm_ncc = confusion_matrix(y_test, y_pred_ncc)
print("NCC confusion matrix:\n", cm_ncc)

results_glass.append({
    "name": f"GLASS_NCC_best_{ncc_grid.best_params_}",
    "train_time": ncc_time,
    "test_accuracy": ncc_test_acc,
    "test_f1_macro": ncc_test_f1_macro,
    "test_f1_weighted": ncc_test_f1_weighted,
})


In [ ]:
# visualize confusion matrix
plt.figure(figsize=(6, 5))
plt.imshow(cm_ncc, interpolation="nearest")
plt.title("NCC Confusion Matrix (Glass)")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.colorbar()
plt.tight_layout()
plt.show()


Summary

In [ ]:
print("\n=== Summary kNN / NCC on glass ===")
for r in results_glass:
    print(
        f"{r['name']}\n"
        f"  train_time       = {r['train_time']:.4f} sec\n"
        f"  test_acc         = {r['test_accuracy']:.4f}\n"
        f"  test_F1_macro    = {r['test_f1_macro']:.4f}\n"
        f"  test_F1_weighted = {r['test_f1_weighted']:.4f}\n"
    )
